# ⬢ Day 20 · Hexagons: 45 years of cities, in bestagons

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2020%20Hexagons/tutorial/day20_cities_in_hexagons.ipynb)

Satellites have photographed Uzbekistan since the 1970s. From those pictures, scientists mapped every place covered by **buildings**. Today you pour that data into **hexagons** and watch the cities grow.

In this notebook you will:
1. meet **H3**, a grid of hexagons that covers the whole Earth,
2. cover Uzbekistan with 10,000 hexagons,
3. drop millions of satellite pixels into them,
4. map how the cities grew from 1975 to 2020,
5. build a **3D map** where every hexagon is a tower,
6. zoom into Tashkent with smaller hexagons, and save a poster.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

**h3** is the hexagon library, and **pydeck** draws 3D maps. The data is **GHSL** (*Global Human Settlement Layer*) from the European Commission: for every small square of land, how many m² are covered by buildings, measured from Landsat and Sentinel-2 satellite images.

We cut it to Uzbekistan and saved two sets of files:
- `built_YEAR.tif`: the whole country, pixels about **1 km** wide,
- `tashkent_built_YEAR.tif`: Tashkent only, pixels about **100 m** wide.

In [ ]:
!pip -q install h3 pydeck

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import h3
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from shapely.geometry import Polygon
from urllib.request import urlretrieve

BASE = "https://raw.githubusercontent.com/NodiraTillayeva/30DayMapChallengeUzbekistan/main/2026/Day%2020%20Hexagons/tutorial/data/"

YEARS = [1975, 1990, 2000, 2010, 2020]
for year in YEARS:
    urlretrieve(BASE + f"built_{year}.tif", f"built_{year}.tif")
    urlretrieve(BASE + f"tashkent_built_{year}.tif", f"tashkent_built_{year}.tif")

regions = gpd.read_file(BASE + "uzbekistan_regions.geojson")
uzbekistan = regions.union_all()
print("Ready! ✅ h3 version", h3.__version__)

## Step 1 · One place, many hexagons

**H3** was made by Uber to count cars and riders. It cuts the whole Earth into hexagons at **16 sizes**, called **resolutions**: 0 is the biggest (a hexagon bigger than Uzbekistan), 15 is the smallest (smaller than a table).

Every hexagon has a name, its **ID**. `h3.latlng_to_cell` tells you which hexagon a point falls into. Let's ask for the hexagons around the **Registan** in Samarkand.

In [ ]:
lat, lng = 39.6548, 66.9757     # the Registan, Samarkand

for res in range(4, 10):
    cell = h3.latlng_to_cell(lat, lng, res)
    area = h3.cell_area(cell, unit="km^2")
    print(f"resolution {res}: {cell}   {area:9.2f} km²")

Each step down makes the hexagons about **7 times smaller**: one hexagon has **7 children** at the next resolution. They fit inside it **almost** perfectly: a hexagon can't be cut into smaller hexagons exactly, so the children stick out a little at the edges. Let's draw the resolution-6 hexagon of the Registan (dark outline), its 7 children at resolution 7 (colours), and their 49 children at resolution 8 (thin lines).

`h3.cell_to_children` finds the children. `h3.cell_to_boundary` gives the six corners of a hexagon, and `hex_shape` turns them into a shape we can draw.

In [ ]:
def hex_shape(cell):
    return Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(cell)])

parent = h3.latlng_to_cell(lat, lng, 6)
children = h3.cell_to_children(parent, 7)
grandchildren = h3.cell_to_children(parent, 8)

fig, ax = plt.subplots(figsize=(7, 7))
gpd.GeoSeries([hex_shape(c) for c in children]).plot(ax=ax, color=["#f1dfb8", "#e9b872", "#d9772f", "#f1dfb8", "#e9b872", "#d9772f", "#a8361e"])
gpd.GeoSeries([hex_shape(c) for c in grandchildren]).boundary.plot(ax=ax, color="white", linewidth=0.8)
gpd.GeoSeries([hex_shape(parent)]).boundary.plot(ax=ax, color="#4a1022", linewidth=3)
ax.plot(lng, lat, "k*", markersize=14)
ax.set_aspect(1.3)
ax.axis("off")
plt.show()

## Step 2 · Cover Uzbekistan with hexagons

`h3.geo_to_cells` fills a shape with all the hexagons whose centre is inside it. We use **resolution 6**: here each hexagon is about **40 km²**, as big as a small town.

Why hexagons and not squares? A square has 4 neighbours across its sides and 4 more at its corners, and the corner ones are further away. A hexagon has **6 neighbours, all at the same distance**. `h3.grid_disk` finds them.

In [ ]:
RES = 6
cells = h3.geo_to_cells(uzbekistan, RES)
print(len(cells), "hexagons cover Uzbekistan")

grid = gpd.GeoDataFrame(geometry=[hex_shape(c) for c in cells], index=cells, crs=4326)

home = h3.latlng_to_cell(41.31, 69.28, RES)          # the centre of Tashkent
ring = h3.grid_disk(home, 1)                         # the hexagon and its 6 neighbours

fig, ax = plt.subplots(figsize=(8, 8))
grid.boundary.plot(ax=ax, color="#bbbbbb", linewidth=0.6)
grid.loc[list(ring)].plot(ax=ax, color="#e9b872")
grid.loc[[home]].plot(ax=ax, color="#a8361e")
ax.set_xlim(68.6, 70.2)
ax.set_ylim(40.8, 41.8)
ax.set_aspect(1.3)
ax.axis("off")
plt.show()

## Step 3 · Pour the pixels into the hexagons

Our country file has **1032 × 2088** pixels. Each one holds the m² of buildings inside it. To fill the hexagons we do three things:
1. find the **centre** (longitude and latitude) of every pixel that has buildings,
2. ask H3 which hexagon that centre falls into,
3. **add up** the m² of all pixels in the same hexagon.

Then we turn m² into **%**: how much of the hexagon is covered by buildings.

In [ ]:
def to_hexagons(path, res):
    with rasterio.open(path) as src:
        built = src.read(1)
        t = src.transform
    rows, cols = np.nonzero(built)                    # only the pixels with buildings
    lngs = t.c + (cols + 0.5) * t.a                   # pixel centres
    lats = t.f + (rows + 0.5) * t.e
    hexes = [h3.latlng_to_cell(la, ln, res) for la, ln in zip(lats, lngs)]
    m2 = pd.Series(built[rows, cols], dtype=float).groupby(hexes).sum()
    area = [h3.cell_area(c, unit="m^2") for c in m2.index]      # each hexagon's own area
    return m2 / area * 100

built = pd.DataFrame({year: to_hexagons(f"built_{year}.tif", RES) for year in YEARS})
built = built.reindex(list(cells)).fillna(0)          # keep only Uzbekistan's hexagons
built.round(1).sort_values(2020, ascending=False).head()

Each row is one hexagon, each column one year. The top hexagons are the centres of **Andijan**, **Namangan** and **Tashkent**: in 2020 about 40% of each is covered by buildings. The rest is streets, yards, gardens and parks.

## Step 4 · Uzbekistan in hexagons, 2020

Now we join the numbers to the hexagon shapes in `grid` and colour them. Empty hexagons are grey, and so are hexagons with less than **0.3%** buildings (one small village). Every hexagon above **15%** gets the darkest colour.

In [ ]:
hexmap = grid.join(built)
city = LinearSegmentedColormap.from_list("city", ["#f1dfb8", "#e9b872", "#d9772f", "#a8361e", "#4a1022"])

def hex_map(year, ax, east_only=False):
    ax.set_facecolor("#f4f1ea")
    hexmap.plot(ax=ax, color="#e6e1d6", linewidth=0)
    some = hexmap[hexmap[year] >= 0.3]
    some.plot(ax=ax, column=year, cmap=city, norm=PowerNorm(0.6, vmin=0, vmax=15), linewidth=0)
    regions.boundary.plot(ax=ax, color="white", linewidth=0.6)
    if east_only:
        ax.set_xlim(64, 73.2)
        ax.set_ylim(37.2, 42.4)
    ax.set_aspect(1.3)
    ax.axis("off")

fig, ax = plt.subplots(figsize=(14, 8), facecolor="#f4f1ea")
hex_map(2020, ax)
plt.show()

## Step 5 · Five maps, 45 years

The same map for every year. We zoom into the east, where most people live.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor="#f4f1ea")
for ax, year in zip(axes.flat, YEARS):
    hex_map(year, ax, east_only=True)
    ax.set_title(year, fontsize=16)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## Step 6 · How much did each region grow?

`sjoin` puts every hexagon into the region its centre falls in. Then we add up the built-up km² per region in 1975 and in 2020.

In [ ]:
area = [h3.cell_area(c, unit="km^2") for c in built.index]
km2 = built.mul(area, axis=0) / 100                                  # back from % to km²
points = [h3.cell_to_latlng(c) for c in km2.index]                # the centre of every hexagon
centres = gpd.GeoDataFrame(km2, geometry=gpd.points_from_xy([p[1] for p in points], [p[0] for p in points]), crs=4326)
centres = centres.sjoin(regions[["region", "geometry"]])

growth = centres.groupby("region")[YEARS].sum()
growth["times"] = growth[2020] / growth[1975]
print("All of Uzbekistan:", round(km2[1975].sum()), "km² →", round(km2[2020].sum()), "km²")
growth.round(1).sort_values("times", ascending=False)

## Step 7 · Hexagons in 3D

**pydeck** knows H3: give it a table with hexagon IDs, and it draws them. Every hexagon becomes a **tower** as tall as its buildings in 2020. Drag with the mouse to move, scroll to zoom, and hold **Ctrl** (or right-click) and drag to tilt and turn. Point at a tower to see its numbers.

`elevation_scale=2500` makes every 1% of buildings 2.5 km tall. One trap: pydeck reads a column name like `"2020"` as the **number** 2020, and every tower becomes 5,000 km tall. So we give our columns names that start with letters: `in_1975`, `in_2020`.

In [ ]:
import pydeck as pdk
from IPython.display import HTML

some = built[built[2020] >= 0.3]
towers = pd.DataFrame({"hex": some.index, "in_1975": some[1975].round(1), "in_2020": some[2020].round(1)})
towers["colour"] = [[int(255 * r), int(255 * g), int(255 * b)]
                    for r, g, b, a in city(PowerNorm(0.6, vmin=0, vmax=15)(towers["in_2020"]))]

layer = pdk.Layer(
    "H3HexagonLayer", towers,
    get_hexagon="hex", get_fill_color="colour",
    extruded=True, get_elevation="in_2020", elevation_scale=2500,
    pickable=True,
)
view = pdk.ViewState(latitude=40.3, longitude=69.0, zoom=6.6, pitch=55, bearing=-20)
deck = pdk.Deck(layers=[layer], initial_view_state=view, map_style="light",
                tooltip={"text": "{hex}\n1975: {in_1975}%\n2020: {in_2020}%"})
HTML(deck.to_html(as_string=True))

## Step 8 · Tashkent with smaller hexagons

For the whole country we used hexagons of about 40 km². For one city we can go smaller. The Tashkent files have 100 m pixels, so we use **resolution 8**: hexagons of less than **1 km²**, 7 × 7 = 49 times smaller.

In [ ]:
RES_CITY = 8
tashkent = pd.DataFrame({year: to_hexagons(f"tashkent_built_{year}.tif", RES_CITY) for year in [1975, 2020]}).fillna(0)
tash_map = gpd.GeoDataFrame(tashkent, geometry=[hex_shape(c) for c in tashkent.index], crs=4326)

def city_map(year, ax):
    ax.set_facecolor("#f4f1ea")
    some = tash_map[tash_map[year] >= 1]
    some.plot(ax=ax, column=year, cmap=city, norm=PowerNorm(0.7, vmin=0, vmax=60), linewidth=0)
    ax.set_xlim(69.03, 69.57)
    ax.set_ylim(41.13, 41.50)
    ax.set_aspect(1.33)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor="#f4f1ea")
for ax, year in zip(axes, [1975, 2020]):
    city_map(year, ax)
    ax.set_title(year, fontsize=16)
plt.tight_layout()
plt.show()

**Small hexagons fit into big ones.** Every hexagon knows its **parent**, the bigger hexagon it belongs to, one or more resolutions up. `h3.cell_to_parent` finds it. The Tashkent hexagons at resolution 8 roll up into the same resolution-6 hexagons as in Step 3.

In [ ]:
small = h3.latlng_to_cell(41.31, 69.28, 8)
print("small hexagon: ", small)
print("its parent:    ", h3.cell_to_parent(small, 6))
print("its children:  ", len(h3.cell_to_children(h3.cell_to_parent(small, 6), 8)), "hexagons at resolution 8")

## Step 9 · Save your poster

Tashkent in 1975 and in 2020, without titles, saved as **tashkent_hexagons.png**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor="#f4f1ea")
for ax, year in zip(axes, [1975, 2020]):
    city_map(year, ax)
plt.savefig("tashkent_hexagons.png", dpi=200, bbox_inches="tight")

from google.colab import files
files.download("tashkent_hexagons.png")

## 🎉 Done!

Share your map with **#30DayMapChallenge**.

**Credits:** GHSL built-up surface (GHS-BUILT-S R2023A), European Commission, Joint Research Centre. Pesaresi M., Politis P. (2023). Hexagons: H3 by Uber.